# Task 2 — Incremental CPG Parser Service

## Approach and reasoning

### Choice of parsing library
The lab allows Joern, tree-sitter, or the standard-library `ast` module. We chose
**`ast`** deliberately:

| | Joern | tree-sitter | `ast` |
|---|---|---|---|
| CPG semantics built in | yes | no | no |
| Extra runtime (JVM / native grammar) | yes | yes | no |
| Deterministic across machines | version-dependent | grammar-dependent | tied to the Python version |
| Control over node identity | limited | manual | full |

Because the grade depends on **reproducible identifiers**, full control over how
identity is computed mattered more to us than getting CPG semantics for free.
`ast` gives an exact, documented tree with no external dependency to pin.

### What we extract
- **AST nodes** and parent→child **AST edges** — the syntactic backbone.
- **CFG edges** — sequential statement flow within a block, plus entry edges into
  `if` / `for` / `while` / `try` / function bodies.
- **DFG edges** — a reaching-definition approximation: a `Store` of a name links
  to later `Load`s of that name in the same scope; parameters count as
  definitions.
- **CALL edges** — each call site to its callee, resolved to a `FunctionDef` in
  the same file when possible, otherwise to a synthetic `ExternalSymbol` node.

CFG and DFG are **intra-procedural approximations**, and we say so explicitly:
the lab asks us to represent these edge categories, not to reimplement a
compiler's dataflow analysis.

### The decision the whole lab rests on: stable identifiers

A naive identifier is `hash(file, line, column)`. It is fatal. Insert one blank
line at the top of a file and every line number below shifts, so every node gets
a new id on replay and Neo4j fills with duplicates — Task 6 fails.

Our identifiers are **structural**:

```
node_id = sha1( file_id | structural_path )
structural_path = chain of (ast_type, field_name, sibling_index) from the root
```

Line and column are still emitted, but only as *properties*, never as identity.
Every element additionally carries the file's sha256 as `file_hash`, which acts
as a **generation marker** for the stale-node sweep in Task 6.

In [1]:
import os, sys
os.chdir("..")
sys.path.insert(0, "src/parser")
print("cwd:", os.getcwd())

cwd: /home/claude/lab04


### Demonstrate the extractor on a small synthetic function

In [2]:
from cpg import extract_cpg

demo = '''
def add(a, b):
    total = a + b
    return total

def main():
    x = add(1, 2)
    print(x)
'''
res = extract_cpg("demo.py", demo)
print("nodes:", len(res.nodes), " edges:", len(res.edges))

from collections import Counter
print("edge types:", Counter(e["type"] for e in res.edges))

nodes: 34  edges: 43
edge types: Counter({'AST': 32, 'CFG': 5, 'DFG': 4, 'CALL': 2})


Look at the actual CALL and DFG edges the extractor found:

In [3]:
for e in res.edges:
    if e["type"] == "CALL":
        print("CALL  ->", e["callee"])
for e in res.edges[:0] or []:
    pass
dfg = [e for e in res.edges if e["type"] == "DFG"]
print("\nDFG edges (def -> use), first 5:")
for e in dfg[:5]:
    print("  variable", e["var"], ":", e["src_id"][:8], "->", e["dst_id"][:8])

CALL  -> add
CALL  -> print

DFG edges (def -> use), first 5:
  variable a : e909c1de -> e2c3cb72
  variable b : 12bd428a -> 3e443051
  variable total : 45ce00a0 -> 64e5ad34
  variable x : 47df68be -> 2358d161


### Proof that identifiers are line-independent

This is the single most important cell in the notebook. We prepend a comment
line to the source — shifting every line number — and check that the node
identifiers are unchanged.

In [4]:
shifted = "# a comment inserted at the top\n" + demo
res2 = extract_cpg("demo.py", shifted)

ids1 = {n["id"] for n in res.nodes}
ids2 = {n["id"] for n in res2.nodes}
print("node ids before :", len(ids1))
print("node ids after  :", len(ids2))
print("identical       :", ids1 == ids2)
print("line numbers changed:",
      res.nodes[3]["start_line"], "->", res2.nodes[3]["start_line"])

node ids before : 34
node ids after  : 34
identical       : True
line numbers changed: 2 -> 3


### Run the Parser Service over the whole repository (offline sink)

In [5]:
!python src/parser/parser_service.py --manifest reports/file_manifest.json \
    --repo ./optimum --offline --outdir reports/events

  processed 10/59 files


  processed 20/59 files


  processed 30/59 files


  processed 40/59 files


  processed 50/59 files


  processed 59/59 files
Event counts: {'node': 58817, 'edge': 73587, 'metadata': 59, 'error': 0}


### Aggregate statistics across all files

In [6]:
import json
from collections import Counter
rows = [json.loads(l)["value"] for l in open("reports/events/cpg.metadata.jsonl")]
ec = Counter()
for r in rows:
    ec.update(r["edge_counts"])
print("files parsed   :", len(rows))
print("lines of code  :", sum(r["loc"] for r in rows))
print("CPG nodes      :", sum(r["num_ast_nodes"] for r in rows))
print("CPG edges      :", sum(r["num_edges"] for r in rows))
print("functions      :", sum(r["num_functions"] for r in rows))
print("classes        :", sum(r["num_classes"] for r in rows))
print("edge breakdown :", dict(ec))

files parsed   : 59
lines of code  : 13725
CPG nodes      : 58817
CPG edges      : 73587
functions      : 522
classes        : 153
edge breakdown : {'AST': 57760, 'CFG': 4987, 'DFG': 8259, 'CALL': 2581}


### A real Kafka message, exactly as it will be produced

In [7]:
import json
sample = json.loads(open("reports/events/samples/cpg.nodes.sample.jsonl").readline())
print(json.dumps(sample, indent=2)[:900])

{
  "key": "b295fb53090cf376624df61f",
  "value": {
    "schema_version": "1.0.0",
    "event_type": "node",
    "event_time": "2026-07-16T13:54:06.989158+00:00",
    "file_id": "e874a4257ccdb966",
    "rel_path": "docs/combine_docs.py",
    "file_hash": "44654054ebb70e05bd2347ed2e38fcd3dd1b203364593391dabc1769ab26f1ee",
    "node": {
      "id": "b295fb53090cf376624df61f",
      "type": "Module",
      "label": "",
      "code": "",
      "start_line": null,
      "end_line": null,
      "scope_id": "b295fb53090cf376624df61f"
    }
  }
}


## Reflection

**What failed.** Our first DFG implementation produced **zero** edges. The bug:
we collected the stable node id at one point but then looked it up again in a
map keyed by Python object id, so every lookup missed. The fix was to store the
stable id directly at collection time. After the fix the same repository yielded
8,258 DFG edges. This is exactly the class of bug that a "the code runs, so it
works" check misses — the pipeline was green while producing an incomplete graph.

**What worked.** Deriving identity from tree structure instead of position. The
cell above proves it: shifting every line in the file left all node identifiers
untouched.

**Known limitation.** DFG is scope-based, not path-sensitive: a variable
reassigned inside an `if` branch links from both definitions to every later use.
Making it path-sensitive would require building a proper CFG-dominator analysis,
which is beyond this lab's scope.